# Standalone automatic structural LVA

This notebook uses only your point CSV, structural trend mesh and modeling parameters. It does **not** read Leapfrog projects, decoded labels, benchmark case folders, or `point_clusters.csv`. The automatic SubDomainer creates and merges its own structural centroids, then uses the recovered Leapfrog-compatible value preprocessing and post-cluster domain rules.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyvista as pv
from IPython.display import display

import polatory
from polatory import three as p3

print("Python package:", polatory.__file__)
print("Automatic builder available:", hasattr(polatory, "AutomaticStructuralDomainBuilder3"))


## Configuration

Change only this cell for another dataset, strength, range or model. The CSV must contain one row per interpolation point and a binary indicator column with positive and negative values.


In [ ]:
DATA_CSV = Path(r"D:\\path\\to\\points.csv")
TREND_MESH_OBJ = Path(r"D:\\path\\to\\trend_mesh.obj")
OUTPUT_OBJ = Path(r"D:\\path\\to\\automatic_lva_result.obj")

X_COLUMN = "x"
Y_COLUMN = "y"
Z_COLUMN = "z"
INDICATOR_COLUMN = "SDF"

# Structural trend parameters: arbitrary positive values are supported.
STRENGTH = 5.0
TREND_RANGE = 100.0

# RBF model.
SILL = 100.0
BASE_RANGE = 400.0
NUGGET = 0.0
POLY_DEGREE = 0
OUTSIDE_VALUE = -1.0
BLEND_POWER = 1.0
MAX_ITERATIONS = 100

# Automatic SubDomainer controls. These mirror the recovered Leapfrog UI defaults.
CENTROID_COUNT = 6000
MINIMUM_CLUSTER_FRACTION = 0.001
MAXIMUM_CLUSTER_FRACTION = 0.10
CONSISTENCY_THRESHOLD = 0.60

# Mesh generation.
ISOSURFACE_RESOLUTION = 25.0
ISOSURFACE_REFINE = 1
BBOX_PADDING_FRACTION = 0.10

# Set both to 3-vectors to override the automatic data bounds.
BBOX_MIN_OVERRIDE = None
BBOX_MAX_OVERRIDE = None

if STRENGTH <= 0 or TREND_RANGE <= 0 or BASE_RANGE <= 0:
    raise ValueError("STRENGTH, TREND_RANGE and BASE_RANGE must be positive")
if not DATA_CSV.exists():
    raise FileNotFoundError(DATA_CSV)
if not TREND_MESH_OBJ.exists():
    raise FileNotFoundError(TREND_MESH_OBJ)


In [ ]:
def read_obj_triangles(path: Path):
    vertices = []
    faces = []
    with path.open("r", encoding="utf-8", errors="ignore") as handle:
        for line in handle:
            if line.startswith("v "):
                vertices.append([float(value) for value in line.split()[1:4]])
            elif line.startswith("f "):
                indices = [int(value.split("/")[0]) - 1 for value in line.split()[1:]]
                if len(indices) == 3:
                    faces.append(indices)
                elif len(indices) > 3:
                    for offset in range(1, len(indices) - 1):
                        faces.append([indices[0], indices[offset], indices[offset + 1]])
    vertices = np.asarray(vertices, dtype=float)
    faces = np.asarray(faces, dtype=np.int64)
    if vertices.ndim != 2 or vertices.shape[1] != 3 or len(vertices) == 0:
        raise ValueError(f"No OBJ vertices found in {path}")
    if faces.ndim != 2 or faces.shape[1] != 3 or len(faces) == 0:
        raise ValueError(f"No triangular OBJ faces found in {path}")
    return vertices, faces

frame = pd.read_csv(DATA_CSV)
required_columns = [X_COLUMN, Y_COLUMN, Z_COLUMN, INDICATOR_COLUMN]
missing = [column for column in required_columns if column not in frame.columns]
if missing:
    raise ValueError(f"Missing CSV columns: {missing}")

points = frame[[X_COLUMN, Y_COLUMN, Z_COLUMN]].to_numpy(dtype=float)
raw_indicators = frame[INDICATOR_COLUMN].to_numpy(dtype=float)
trend_vertices, trend_faces = read_obj_triangles(TREND_MESH_OBJ)

if len(points) < 2 or not np.all(np.isfinite(points)):
    raise ValueError("Point coordinates must contain at least two finite rows")
if not (np.any(raw_indicators > 0) and np.any(raw_indicators < 0)):
    raise ValueError("The indicator column must contain positive and negative classes")

value_info = polatory.leapfrog_indicator_values3(points, raw_indicators)
values = value_info.values
FIT_TOLERANCE = value_info.fit_accuracy

print(f"Input points: {len(points):,}")
print(f"Trend mesh: {len(trend_vertices):,} vertices, {len(trend_faces):,} triangles")
print("Data diagonal:", value_info.data_diagonal)
print("Automatic fit tolerance:", FIT_TOLERANCE)
print("Processed value range:", float(values.min()), float(values.max()))


## Build automatic domains

This cell calculates the LVA field, generates the centroid grid, performs deterministic neighbouring-region merges, assigns every interpolation point to an automatic domain, and applies the exact recovered post-cluster support/range rules.


In [ ]:
trend_input = polatory.StructuralTrendInput3(
    trend_vertices,
    trend_faces,
    float(STRENGTH),
    float(TREND_RANGE),
)

rbf = p3.CovSpheroidal3([SILL, BASE_RANGE])
model = p3.Model(rbf, POLY_DEGREE)
model.nugget = NUGGET
model_parameters = np.asarray(model.parameters, dtype=float).reshape(-1).tolist()

builder = polatory.AutomaticStructuralDomainBuilder3(
    centroid_count=CENTROID_COUNT,
    minimum_cluster_fraction=MINIMUM_CLUSTER_FRACTION,
    maximum_cluster_fraction=MAXIMUM_CLUSTER_FRACTION,
    consistency_threshold=CONSISTENCY_THRESHOLD,
    base_range=BASE_RANGE,
    minimum_support_points=1,
)
domains = builder.build_from_inputs(
    points,
    [trend_input],
    model_parameters=model_parameters,
)
labels = builder.labels_
diagnostics = builder.diagnostics_

domain_table = pd.DataFrame([
    {
        "domain": item.label,
        "core_points": len(item.core_indices),
        "support_points": len(item.support_indices),
        "local_kernel_range": item.local_kernel_range,
        "internal_radius": item.internal_radius,
    }
    for item in diagnostics.postcluster
])

print("Centroid grid shape:", diagnostics.centroid_grid_shape)
print("Centroids:", len(diagnostics.centroid_points))
print("Automatic merge operations:", diagnostics.merge_count)
print("Final automatic domains:", diagnostics.final_domain_count)
print("Resolved point limits:", diagnostics.minimum_points, diagnostics.maximum_points)
display(domain_table)


## Inspect the automatic partition and LVA field


In [ ]:
point_cloud = pv.PolyData(points)
point_cloud["automatic_domain"] = labels
centroid_cloud = pv.PolyData(diagnostics.centroid_points)
centroid_cloud["automatic_domain"] = diagnostics.centroid_labels

vtk_faces = np.hstack([
    np.full((len(trend_faces), 1), 3, dtype=np.int64),
    trend_faces,
]).ravel()
trend_surface = pv.PolyData(trend_vertices, vtk_faces)

plotter = pv.Plotter()
plotter.add_mesh(centroid_cloud, scalars="automatic_domain", point_size=3, render_points_as_spheres=True, opacity=0.30)
plotter.add_mesh(point_cloud, scalars="automatic_domain", point_size=8, render_points_as_spheres=True)
plotter.add_mesh(trend_surface, opacity=0.25, show_edges=True)
plotter.show_grid()
plotter.show()


In [ ]:
data_min = points.min(axis=0)
data_max = points.max(axis=0)
data_span = data_max - data_min
safe_span = np.where(data_span > 0, data_span, max(np.linalg.norm(data_span), 1.0) * 0.05)

lva_min = data_min - 0.05 * safe_span
lva_max = data_max + 0.05 * safe_span
lva_dimensions = (25, 25, 25)
axes = [np.linspace(lva_min[i], lva_max[i], lva_dimensions[i]) for i in range(3)]
xx, yy, zz = np.meshgrid(*axes, indexing="ij")
lva_grid = pv.StructuredGrid(xx, yy, zz)
lva_matrices = polatory.sample_single_input_anisotropies3(lva_grid.points, trend_input)
eigenvalues, eigenvectors = np.linalg.eigh(lva_matrices)
lva_ratio = eigenvalues[:, -1] / eigenvalues[:, 0]
lva_grid["LVA ratio"] = lva_ratio
centre = 0.5 * (lva_min + lva_max)
slices = lva_grid.slice_orthogonal(x=centre[0], y=centre[1], z=centre[2])

plotter = pv.Plotter()
plotter.add_mesh(slices, scalars="LVA ratio", clim=(1.0, max(1.01, STRENGTH)), opacity=0.85)
plotter.add_mesh(trend_surface, opacity=0.30, show_edges=True)
plotter.add_mesh(point_cloud, point_size=5, render_points_as_spheres=True)
plotter.show_grid()
plotter.show()
print("LVA ratio range:", float(lva_ratio.min()), float(lva_ratio.max()))


## Fit and export the implicit surface


In [ ]:
structural = polatory.StructuralInterpolant3(
    model,
    outside_value=OUTSIDE_VALUE,
    blend_power=BLEND_POWER,
)
structural.fit(
    points,
    values,
    domains,
    tolerance=FIT_TOLERANCE,
    max_iter=MAX_ITERATIONS,
)

training_predictions = structural.evaluate(points)
training_errors = training_predictions - values
print("Training RMSE:", float(np.sqrt(np.mean(training_errors**2))))
print("Training maximum absolute error:", float(np.max(np.abs(training_errors))))

if BBOX_MIN_OVERRIDE is None or BBOX_MAX_OVERRIDE is None:
    minimum = points.min(axis=0)
    maximum = points.max(axis=0)
    span = maximum - minimum
    fallback = max(float(np.linalg.norm(span)), BASE_RANGE, 1.0)
    padding = BBOX_PADDING_FRACTION * np.where(span > 0.0, span, fallback)
    bbox_min = minimum - padding
    bbox_max = maximum + padding
else:
    bbox_min = np.asarray(BBOX_MIN_OVERRIDE, dtype=float)
    bbox_max = np.asarray(BBOX_MAX_OVERRIDE, dtype=float)

bbox = p3.Bbox(bbox_min.reshape(1, 3), bbox_max.reshape(1, 3))
field = polatory.StructuralRbfFieldFunction(structural)
result_mesh = polatory.Isosurface(
    bbox,
    ISOSURFACE_RESOLUTION,
    np.eye(3),
).generate(field, isovalue=0.0, refine=ISOSURFACE_REFINE)

OUTPUT_OBJ.parent.mkdir(parents=True, exist_ok=True)
result_mesh.export_obj(str(OUTPUT_OBJ))
print("Exported:", OUTPUT_OBJ)


In [ ]:
generated = pv.read(OUTPUT_OBJ).extract_surface().triangulate().clean()
plotter = pv.Plotter()
plotter.add_mesh(generated, opacity=0.80, show_edges=True, label="Automatic LVA surface")
plotter.add_mesh(trend_surface, opacity=0.20, label="Structural trend mesh")
plotter.add_mesh(point_cloud, scalars=values, point_size=6, render_points_as_spheres=True, label="Interpolation points")
plotter.add_legend()
plotter.show_grid()
plotter.show()
